# C2 数值复核记录

第 03 章《离散的世界》｜G22 · C2｜第一轮复核草稿｜组员复核日期：2026-11-21｜AI 整理日期：2026-11-23

受审仓库快照 `03bcde0d1b14aebb28cef924f487136c5470026a`。本轮推导与代码由 ChatGPT（Codex）生成，数值由 Python 实际运行；复核分工：徐雯雨负责核对关键问题的推导，蒋友涵负责复跑 Notebook 实验。

依赖 Python 3 与 mpmath 1.3.0。按顺序运行。四项均为另写的最小基准，受审文件见同目录批判报告。

## 判定方法

先用放大因子、离散正弦谱、精确节点载荷及指数解析解给定参考值，再以递推、差分更新、矩阵装配或二分求根检验。断言直接记录容差。

## 3.1　有界、衰减与准确性

取同一衰减方程，直接执行经典 RK4 的四个阶段，并与解析解和放大多项式比较。

In [1]:
from math import exp, sin, cos, pi
def rk_step(y, h, a=1):
    k1 = -a*y
    k2 = -a*(y+h*k1/2)
    k3 = -a*(y+h*k2/2)
    k4 = -a*(y+h*k3)
    return y+h*(k1+2*k2+2*k3+k4)/6

def R(z):
    return 1+z+z*z/2+z**3/6+z**4/24

for h in (2, 3):
    euler = rk = 1.
    for step in range(3):
        euler += -h*euler
        rk = rk_step(rk,h)
    assert abs(rk-R(-h)**3)<1e-12
    print(f'h={h}, Euler={euler:.9f}, RK4={rk:.9f}, exact={exp(-3*h):.9f}')
assert abs(R(-3))>1 and abs(1-2)==1

h=2, Euler=-1.000000000, RK4=0.037037037, exact=0.002478752
h=3, Euler=-8.000000000, RK4=2.599609375, exact=0.000123410


## 3.2　固定网格上的两种稳定要求

单位区间四等分，零 Dirichlet 边界。分别输入最高正弦模态与非负尖峰。

In [2]:
J, r = 4, 0.55
def ftcs(v):
    return [0]+[v[j]+r*(v[j-1]-2*v[j]+v[j+1]) for j in range(1,J)]+[0]
g = [1-4*r*sin(k*pi/(2*J))**2 for k in range(1,J)]
v = [sin((J-1)*pi*j/J) for j in range(J+1)]
v[-1] = 0
w = ftcs(v)
spike = ftcs([0,0,1,0,0])
assert max(abs(x) for x in g)<1
assert max(abs(y-g[-1]*x) for x,y in zip(v,w))<1e-12
assert abs(spike[2]+0.1)<1e-12
print('模态因子', [round(x,9) for x in g], '非负输入的一步结果',spike)

模态因子 [0.677817459, -0.1, -0.877817459] 非负输入的一步结果 [0, 0.55, -0.10000000000000009, 0.55, 0]


## 3.3　四点 Gauss 载荷的高精度复测

在 40、70 位十进制精度下分别装配正弦载荷与刚度矩阵。解析解的节点向量乘刚度矩阵给出精确载荷基准。

In [3]:
import mpmath as mp

def fem_error(N, digits):
    with mp.workdps(digits):
        h = mp.mpf(1)/N
        nodes, weights = mp.gauss_quadrature(4,'legendre')
        K = mp.matrix(N-1)
        F = mp.matrix(N-1,1)
        for e in range(N):
            for side in (0,1):
                j = e+side
                if 0 < j < N:
                    for t,w in zip(nodes,weights):
                        x = h*(e+(1+t)/2)
                        phi = (1+t)/2 if side else (1-t)/2
                        F[j-1] += w*h/2 * mp.pi**2*mp.sin(mp.pi*x)*phi
                    K[j-1,j-1] += 1/h
                    other = e+1-side
                    if 0 < other < N:
                        K[j-1,other-1] -= 1/h
        exact = mp.matrix([mp.sin(mp.pi*j*h) for j in range(1,N)])
        return float(mp.norm(mp.lu_solve(K,F)-exact,mp.inf)), float(mp.norm(F-K*exact,mp.inf))
for N in (4,8,16):
    e40,_ = fem_error(N,40)
    e70,load = fem_error(N,70)
    assert e70>1e-17 and abs(e40-e70)<1e-12*e70
    print(f'N={N}: max node error={e70:.12e}, max load error={load:.12e}')

N=4: max node error=5.863633429909e-10, max load error=1.373934775383e-09
N=8: max node error=2.242228072516e-12, max load error=2.730871185539e-12
N=16: max node error=8.712494875611e-15, max load error=5.357060672741e-15


## 3.4　同一训练轨迹的两组参数

复核原搜索区间，再放宽参数范围。用离散观测与连续解析解分别评价同一参数。

In [4]:
h, target = 0.2, exp(-0.32)
def root(lo, hi):
    def f(a):
        return R(-h*a)-target
    assert f(lo)*f(hi)<0
    for step in range(60):
        mid = (lo+hi)/2
        if f(mid)==0:
            return mid
        if f(lo)*f(mid)<0:
            hi = mid
        else:
            lo = mid
    return (lo+hi)/2
for a in (root(.5,2.5), root(10,15)):
    error = max(abs(y*R(-h*a)**n-y*exp(-1.6*h*n))
                for y in (.8,1,1.2) for n in range(11))
    continuous = abs(exp(-a*h)-target)
    assert error<1e-13
    print(f'a={a:.10f}, max training error={error:.3e}, continuous error at t=0.2 (y0=1)={continuous:.9f}')

a=1.6001828258, max training error=1.110e-16, continuous error at t=0.2 (y0=1)=0.000026551
a=12.8731762097, max training error=1.110e-16, continuous error at t=0.2 (y0=1)=0.649967431


## 复核结论

B2 的 V7 把四点 Gauss 节点误差归为舍入；提高精度仍保留误差。其余三个实验展示适用边界。未运行完整 A/B Notebook。实验复跑人：蒋友涵。复跑日期：2026-11-21。运行环境、结果差异及采纳意见待填写。